# Raising accuracy: stronger models, CLIP fine-tuning, ensembles

Before you start: **Runtime → Change runtime type → T4 GPU** (an A100 or L4 is about 3x faster if you have Colab Pro). Then **Runtime → Run all**. On a T4 it takes about 1.5 to 2 hours.

Same data as before (`My Drive/colab_data.zip`), same artist-disjoint split, same museum test set, balanced accuracy as the main metric. Nothing here looks at test or museum images to make a choice: every setting is picked on **WikiArt val** only.

What it tries, cheapest first:
1. **SigLIP** (a newer, stronger CLIP-like model) as a frozen-feature probe.
2. **Feature ensemble**: CLIP + DINOv2 + SigLIP features joined, one probe, class-balanced.
3. **Fine-tune CLIP ViT-L/14**: the last 8 of 24 blocks are trained (the rest stay frozen), starting from the style names as the classifier, with weight averaging towards the original model (WiSE-FT), which is known to keep out-of-domain accuracy.
4. **Test-time augmentation** (7 views per painting: full, mirrored, 5 crops), averaged.
5. **Final ensemble** of the fine-tuned model and the feature-ensemble probe.

Results go to `My Drive/wikiart_runs/boost/`. The last cell prints one table to paste back.

## 0. Settings (leave as they are for the first run)

In [ ]:
SEED = 1              # repeat with 2 and 3 for the report if time allows
N_TRAIN_BLOCKS = 8    # CLIP blocks to fine-tune (24 = full fine-tune; needs an A100)
EPOCHS = 6
BATCH = 32
QUICK = False         # True = tiny run to check that everything works (about 5 minutes)

## 1. Connect Drive and unpack the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/data/final/wikiart_split.csv'):
    !cp /content/drive/MyDrive/colab_data.zip /content/
    !cd /content && unzip -q -o colab_data.zip && rm colab_data.zip
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Load the sets (same split and museum test set as the baseline)

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
DATA = Path('/content/data')
PROBES = Path('/content/drive/MyDrive/wikiart_runs/probes')      # CLIP/DINOv2 features from the last notebook
OUT = Path('/content/drive/MyDrive/wikiart_runs/boost'); OUT.mkdir(parents=True, exist_ok=True)

split = pd.read_csv(DATA / 'final/wikiart_split.csv')
CLASSES = sorted(split['style'].unique())
museum = pd.read_csv(DATA / 'final/museum_test.csv')
museum = museum[museum['style'].isin(CLASSES)]
sets = {s: split[split['split'] == s] for s in ('train', 'val', 'test')}
sets['met'] = museum[museum['museum'] == 'met']
sets['rijks'] = museum[museum['museum'] == 'rijks']
for k in sets:
    df = sets[k]
    sets[k] = df[df['image'].map(lambda p: (DATA / p).exists())].reset_index(drop=True)
if QUICK:
    sets = {k: df.groupby('style').head(40).reset_index(drop=True) for k, df in sets.items()}
Y = {k: np.array([CLASSES.index(s) for s in df['style']]) for k, df in sets.items()}
print('classes:', CLASSES)
for k, df in sets.items():
    print(f'{k:<6} {len(df):>6} images')
# Expected: train 18501, val 3786, test 3857, met 937, rijks 3092

## 3. Metrics (same as before, plus top-3 and a 3-era grouping)

In [ ]:
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score

def era(c):   # coarse grouping, reported only as an extra
    if 'Mannerism' in c or 'Renaissance' in c: return 0
    if c in ('Baroque', 'Rococo'): return 1
    return 2
ERA = np.array([era(c) for c in CLASSES])

def ece(y, probs, bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return e

def scores(y, probs):
    p = probs.argmax(1)
    return dict(n=len(y), acc=accuracy_score(y, p), bal_acc=balanced_accuracy_score(y, p),
                macro_f1=f1_score(y, p, average='macro'),
                top3=(np.argsort(-probs, 1)[:, :3] == y[:, None]).any(1).mean(),
                era3_bal_acc=balanced_accuracy_score(ERA[y], ERA[p]), ece=ece(y, probs))

def unseen(k):
    return np.ones(len(sets[k]), bool) if k in ('val', 'test') else (sets[k]['artist_group'].values == 'unseen')

RESULTS = {}   # model name -> {set: probs}
def save_preds(name, probs):
    RESULTS[name] = probs
    for k, pr in probs.items():
        out = sets[k][['image', 'style']].copy(); out['pred'] = [CLASSES[i] for i in pr.argmax(1)]
        for i, c in enumerate(CLASSES): out[f'p_{c}'] = pr[:, i]
        if 'artist_group' in sets[k]: out['artist_group'] = sets[k]['artist_group'].values
        out.to_csv(OUT / f'preds_{name}_{k}.csv', index=False)

def show(name):
    pr = RESULTS[name]
    line = f'{name:<22}'
    for k in ('val', 'test', 'met', 'rijks'):
        m = unseen(k); line += f'  {k} {100*balanced_accuracy_score(Y[k][m], pr[k][m].argmax(1)):5.1f}'
    print(line + '   (bal acc; museums = unseen artists)')

## 4. Frozen features: SigLIP (new), plus CLIP and DINOv2 (reused from Drive if there)
About 15 minutes for SigLIP on a T4. Saved to Drive, so it only runs once.

In [ ]:
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, AutoModel, CLIPModel, CLIPProcessor, AutoProcessor
dev = 'cuda'
def vec(o):
    return o if torch.is_tensor(o) else o.pooler_output
MODELS = {'clip': 'openai/clip-vit-large-patch14', 'dinov2': 'facebook/dinov2-large',
          'siglip': 'google/siglip-so400m-patch14-384'}

class Images(Dataset):
    def __init__(self, paths, tf): self.paths, self.tf = paths, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(Image.open(DATA / self.paths[i]).convert('RGB'))

@torch.no_grad()
def extract(name):
    f = OUT / f'quick_features_{name}.npz' if QUICK else PROBES / f'features_{name}.npz'
    if f.exists():
        print(name, 'features already on Drive'); return f
    PROBES.mkdir(parents=True, exist_ok=True)
    if name == 'clip':
        model = CLIPModel.from_pretrained(MODELS[name], torch_dtype=torch.float16).to(dev).eval()
        proc = CLIPProcessor.from_pretrained(MODELS[name]).image_processor
        embed = lambda x: vec(model.get_image_features(pixel_values=x))
    elif name == 'siglip':
        model = AutoModel.from_pretrained(MODELS[name], torch_dtype=torch.float16).to(dev).eval()
        proc = AutoImageProcessor.from_pretrained(MODELS[name])
        embed = lambda x: vec(model.get_image_features(pixel_values=x))
    else:
        model = AutoModel.from_pretrained(MODELS[name], torch_dtype=torch.float16).to(dev).eval()
        proc = AutoImageProcessor.from_pretrained(MODELS[name])
        def embed(x):
            h = model(pixel_values=x).last_hidden_state
            return torch.cat([h[:, 0], h[:, 1:].mean(1)], 1)
    tf = lambda img: proc(images=img, return_tensors='pt')['pixel_values'][0]
    feats = {}
    for k, df in sets.items():
        dl = DataLoader(Images(df['image'].tolist(), tf), batch_size=32, num_workers=2)
        feats[k] = np.concatenate([embed(x.to(dev, torch.float16)).float().cpu().numpy() for x in dl])
        print(f'{name} {k:<6} {feats[k].shape}', flush=True)
    np.savez(f, **feats)
    del model; torch.cuda.empty_cache()
    return f

FEAT = {}
for name in MODELS:
    FEAT[name] = dict(np.load(extract(name)))
    assert all(len(FEAT[name][k]) == len(sets[k]) for k in sets), f'{name}: feature count does not match the sets; delete the old features file and rerun'

## 5. Linear probes: SigLIP alone, and CLIP + DINOv2 + SigLIP joined
Class-balanced (because balanced accuracy is the metric), strength chosen on WikiArt val. About 10 minutes.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def probe(names):
    F = {k: np.concatenate([FEAT[n][k] / np.linalg.norm(FEAT[n][k], axis=1, keepdims=True) for n in names], 1) for k in sets}
    sc = StandardScaler().fit(F['train']); F = {k: sc.transform(v) for k, v in F.items()}
    best = None
    for C in (0.001, 0.003, 0.01, 0.03, 0.1):
        clf = LogisticRegression(C=C, max_iter=3000, class_weight='balanced').fit(F['train'], Y['train'])
        b = balanced_accuracy_score(Y['val'], clf.predict(F['val']))
        print(f'  {"+".join(names)}  C={C:<6} val bal acc {b:.3f}', flush=True)
        if best is None or b > best[0]: best = (b, clf)
    return {k: best[1].predict_proba(F[k]) for k in ('val', 'test', 'met', 'rijks')}

save_preds('clip_probe_bal', probe(['clip']))
save_preds('siglip_probe', probe(['siglip']))
save_preds('feat_ensemble', probe(['clip', 'dinov2', 'siglip']))
for n in ('clip_probe_bal', 'siglip_probe', 'feat_ensemble'): show(n)

## 6. Fine-tune CLIP ViT-L/14 (about 5 to 8 minutes per epoch on a T4)
The best epoch (on WikiArt val) is saved to Drive.

In [ ]:
import math, time, torch.nn as nn, torch.nn.functional as Fn
from torchvision import transforms as TT
torch.manual_seed(SEED); np.random.seed(SEED)
MEAN, STD = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)
train_tf = TT.Compose([TT.RandomResizedCrop(224, scale=(0.5, 1.0)), TT.RandomHorizontalFlip(),
                       TT.ToTensor(), TT.Normalize(MEAN, STD)])
plain_tf = TT.Compose([TT.Resize((224, 224)), TT.ToTensor(), TT.Normalize(MEAN, STD)])
def tta_tf(img):   # 7 views: whole, mirrored, 4 corners + centre
    norm = TT.Compose([TT.ToTensor(), TT.Normalize(MEAN, STD)])
    whole = img.resize((224, 224), Image.BICUBIC)
    crops = TT.FiveCrop(224)(img.resize((256, 256), Image.BICUBIC))
    return torch.stack([norm(v) for v in (whole, TT.functional.hflip(whole), *crops)])

clip = CLIPModel.from_pretrained(MODELS['clip'])
proc = CLIPProcessor.from_pretrained(MODELS['clip'])
with torch.no_grad():   # start the classifier from the style names (zero-shot weights)
    names = [c.replace(' (Late Renaissance)', '') for c in CLASSES]
    tmpl = ['a painting in the style of {}.', 'a {} painting.', 'an artwork from the {} period.']
    W = []
    for n in names:
        t = proc(text=[x.format(n) for x in tmpl], return_tensors='pt', padding=True)
        e = Fn.normalize(vec(clip.get_text_features(**t)), dim=-1); W.append(Fn.normalize(e.mean(0), dim=0))
    W = torch.stack(W)

class StyleCLIP(nn.Module):
    def __init__(self, clip, W):
        super().__init__()
        self.vision, self.proj = clip.vision_model, clip.visual_projection
        self.head = nn.Parameter(W.clone())
        self.log_scale = nn.Parameter(torch.tensor(math.log(30.0)))
    def forward(self, x):
        f = Fn.normalize(self.proj(self.vision(pixel_values=x).pooler_output), dim=-1)
        return self.log_scale.exp() * f @ Fn.normalize(self.head, dim=-1).t()

model = StyleCLIP(clip, W); del clip
for p in model.parameters(): p.requires_grad = False
blocks = model.vision.encoder.layers
groups, BASE_LR, DECAY = [], 1e-5, 0.8
for depth, blk in enumerate(reversed(blocks[-N_TRAIN_BLOCKS:])):   # lower blocks get smaller steps
    for p in blk.parameters(): p.requires_grad = True
    groups.append(dict(params=list(blk.parameters()), lr=BASE_LR * DECAY ** depth))
top = list(model.vision.post_layernorm.parameters()) + list(model.proj.parameters())
for p in top + [model.head, model.log_scale]: p.requires_grad = True
groups += [dict(params=top, lr=BASE_LR), dict(params=[model.head, model.log_scale], lr=1e-3, weight_decay=0.0)]
for g in groups: g['initial_lr'] = g['lr']
model = model.to(dev)
trainable = lambda: {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
ZERO_SHOT = trainable()
print(f'trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.0f} M')

counts = np.bincount(Y['train'], minlength=len(CLASSES))
weights = torch.tensor(len(Y['train']) / (len(CLASSES) * counts), dtype=torch.float32, device=dev)
loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

class Labeled(Dataset):
    def __init__(self, k, tf): self.paths, self.y, self.tf = sets[k]['image'].tolist(), Y[k], tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(Image.open(DATA / self.paths[i]).convert('RGB')), self.y[i]

@torch.no_grad()
def predict(k, tta=False):
    model.eval(); out = []
    dl = DataLoader(Labeled(k, tta_tf if tta else plain_tf), batch_size=16 if tta else 64, num_workers=2)
    for x, _ in dl:
        x = x.to(dev)
        with torch.autocast('cuda', dtype=torch.float16):
            if tta:
                b, v = x.shape[:2]; pr = model(x.flatten(0, 1)).float().softmax(-1).view(b, v, -1).mean(1)
            else:
                pr = model(x).float().softmax(-1)
        out.append(pr.cpu().numpy())
    return np.concatenate(out)

train_dl = DataLoader(Labeled('train', train_tf), batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
EP = 1 if QUICK else EPOCHS
opt = torch.optim.AdamW(groups, weight_decay=0.05)
total, warm = EP * len(train_dl), len(train_dl) // 2
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
scaler = torch.amp.GradScaler('cuda')
CKPT = OUT / f'clip_ft_seed{SEED}_best.pt'
best = (-1, None)
for ep in range(EP):
    model.train(); t0, run = time.time(), 0.0
    for i, (x, y) in enumerate(train_dl):
        with torch.autocast('cuda', dtype=torch.float16):
            loss = loss_fn(model(x.to(dev)).float(), y.to(dev))
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_([p for g in groups for p in g['params']], 1.0)
        scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
    v = balanced_accuracy_score(Y['val'], predict('val').argmax(1))
    print(f'epoch {ep+1}/{EP}  loss {run/len(train_dl):.3f}  val bal acc {v:.3f}  ({(time.time()-t0)/60:.1f} min)', flush=True)
    if v > best[0]:
        best = (v, trainable()); torch.save(best[1], CKPT)
print(f'best val bal acc {best[0]:.3f}, saved to {CKPT}')

## 7. WiSE-FT: blend the fine-tuned weights with the original ones (choice made on WikiArt val)
Blending usually keeps most of the gain and protects accuracy on new domains such as museum photos.

In [ ]:
def set_alpha(a):
    with torch.no_grad():
        for n, p in model.named_parameters():
            if p.requires_grad: p.copy_(((1 - a) * ZERO_SHOT[n] + a * best[1][n]).to(dev))
alphas = {}
for a in (0.5, 0.7, 0.85, 1.0):
    set_alpha(a); alphas[a] = balanced_accuracy_score(Y['val'], predict('val').argmax(1))
    print(f'alpha {a:<5} val bal acc {alphas[a]:.3f}', flush=True)
ALPHA = max(alphas, key=alphas.get); set_alpha(ALPHA)
print('chose alpha', ALPHA)

## 8. Score the fine-tuned model, with and without test-time augmentation (about 10 minutes)

In [ ]:
save_preds(f'clip_ft_s{SEED}', {k: predict(k) for k in ('val', 'test', 'met', 'rijks')})
save_preds(f'clip_ft_tta_s{SEED}', {k: predict(k, tta=True) for k in ('val', 'test', 'met', 'rijks')})
for n in (f'clip_ft_s{SEED}', f'clip_ft_tta_s{SEED}'): show(n)

## 9. Final ensemble: fine-tuned CLIP (with TTA) + feature-ensemble probe, weight chosen on WikiArt val

In [ ]:
A, B = RESULTS[f'clip_ft_tta_s{SEED}'], RESULTS['feat_ensemble']
ws = {w: balanced_accuracy_score(Y['val'], (w * A['val'] + (1 - w) * B['val']).argmax(1)) for w in (0, .25, .5, .75, 1)}
print({w: round(v, 3) for w, v in ws.items()})
w = max(ws, key=ws.get); print('chose weight', w, 'for the fine-tuned model')
save_preds(f'final_ensemble_s{SEED}', {k: w * A[k] + (1 - w) * B[k] for k in A})
show(f'final_ensemble_s{SEED}')

## 10. Summary table (paste this back)
Balanced accuracy in %. Museum columns are unseen artists unless marked seen. "3-era" groups the six styles into Renaissance / Baroque-Rococo / 19th century and is an extra, not the main result.

In [ ]:
rng = np.random.default_rng(0)
def boot(y, p, B=2000):
    n = len(y); return np.array([balanced_accuracy_score(y[i], p[i]) for i in (rng.integers(0, n, n) for _ in range(B))])
rows = []
for name, pr in RESULTS.items():
    r = dict(model=name)
    for k in ('test', 'met', 'rijks'):
        m = unseen(k); s = scores(Y[k][m], pr[k][m])
        r[f'{k} bal acc'] = 100 * s['bal_acc']; r[f'{k} top3'] = 100 * s['top3']; r[f'{k} 3-era'] = 100 * s['era3_bal_acc']
        if k == 'test': r['test macro-F1'] = 100 * s['macro_f1']; r['test ECE'] = s['ece']
    m = sets['met']['artist_group'].values == 'seen'
    r['met seen bal acc'] = 100 * balanced_accuracy_score(Y['met'][m], pr['met'][m].argmax(1))
    rows.append(r)
tab = pd.DataFrame(rows).round(1)
tab.to_csv(OUT / f'boost_summary_s{SEED}.csv', index=False)
pd.set_option('display.width', 250)
print(tab.to_string(index=False))

name = f'final_ensemble_s{SEED}'; pr = RESULTS[name]
pt = pr['test'].argmax(1); bt = boot(Y['test'], pt)
lines = [f'{name}: WikiArt test {100*balanced_accuracy_score(Y["test"], pt):.1f} [{100*np.percentile(bt,2.5):.1f}, {100*np.percentile(bt,97.5):.1f}]']
for k in ('met', 'rijks'):
    m = unseen(k); p = pr[k][m].argmax(1); b = boot(Y[k][m], p); obs = balanced_accuracy_score(Y['test'], pt) - balanced_accuracy_score(Y[k][m], p)
    gap = bt - b; pval = min(1.0, 2 * min((gap <= 0).mean(), (gap >= 0).mean()))
    lines.append(f'  {k} unseen {100*balanced_accuracy_score(Y[k][m], p):.1f} [{100*np.percentile(b,2.5):.1f}, {100*np.percentile(b,97.5):.1f}]'
                 f'   gap {100*obs:.1f} [{100*np.percentile(gap,2.5):.1f}, {100*np.percentile(gap,97.5):.1f}] p = {pval:.3f}')
print('\n' + '\n'.join(lines))
(OUT / f'boost_report_s{SEED}.txt').write_text(tab.to_string(index=False) + '\n\n' + '\n'.join(lines))

## 11. Rijks Mannerism check (every model scores it low; is it a label problem?)
Saves the Rijks paintings labelled Mannerism with the final prediction, so you can look at the artists and dates.

In [ ]:
k = 'rijks'; pr = RESULTS[f'final_ensemble_s{SEED}'][k]
m = sets[k]['style'].str.startswith('Mannerism').values
chk = sets[k][m].copy(); chk['pred'] = [CLASSES[i] for i in pr[m].argmax(1)]; chk['confidence'] = pr[m].max(1).round(2)
chk.to_csv(OUT / 'rijks_mannerism_check.csv', index=False)
print(chk['pred'].value_counts().to_string())
cols = [c for c in chk.columns if any(w in c.lower() for w in ('artist', 'title', 'date', 'year', 'source', 'rule'))]
print(chk[cols + ['pred', 'confidence']].head(40).to_string(index=False))